# FreshRetailNet-50K Analysis
**Dataset**: Dingdong-Inc/FreshRetailNet-50K (HuggingFace)

> 4.5M train rows + 350K eval rows, 19 columns. Includes hourly sales/stock arrays, weather, holidays.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load sample of train + full eval
train = pd.read_csv('abhinav/fresh-retail-net/train.csv', nrows=200000)
eval_df = pd.read_csv('abhinav/fresh-retail-net/eval.csv')

print(f'Train sample: {len(train):,} rows x {train.shape[1]} cols')
print(f'Eval: {len(eval_df):,} rows x {eval_df.shape[1]} cols')
print(f'\nColumns: {train.columns.tolist()}')
train.head()


## 1. Data Quality & Schema

In [ ]:
print('=== Data Types ===')
print(train.dtypes)
print(f'\n=== Null Counts ===')
print(train.isnull().sum())
print(f'\n=== Unique Values ===')
for col in train.columns:
    print(f'{col}: {train[col].nunique()}')


## 2. Statistical Summary

In [ ]:
train.describe()

## 3. Category Hierarchy

In [ ]:
cat_hier = [c for c in train.columns if 'category' in c.lower() or 'first' in c.lower() or 'second' in c.lower() or 'third' in c.lower()]
for col in cat_hier:
    if train[col].dtype == 'object' or train[col].nunique() < 50:
        print(f'\n{col} — {train[col].nunique()} unique')
        print(train[col].value_counts().head(10))


In [ ]:
# Category distribution visualization
cat_cols = [c for c in train.columns if train[c].dtype == 'object' and train[c].nunique() < 30]
if not cat_cols:
    cat_cols = [c for c in train.columns if train[c].nunique() < 20 and train[c].dtype in ['int64','float64']]

n = min(len(cat_cols), 4)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 5*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(cat_cols[:n]):
        train[col].value_counts().head(15).plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
        axes[i].set_title(f'{col}')
        axes[i].tick_params(axis='x', rotation=45)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Sales Analysis

In [ ]:
sale_cols = [c for c in train.columns if 'sale' in c.lower() or 'amount' in c.lower()]
for col in sale_cols:
    if train[col].dtype in ['float64', 'int64']:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.hist(train[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='teal')
        ax.set_title(f'{col} Distribution (mean={train[col].mean():.2f})')
        ax.axvline(train[col].mean(), color='red', linestyle='--', label=f'Mean: {train[col].mean():.2f}')
        ax.axvline(train[col].median(), color='orange', linestyle='--', label=f'Median: {train[col].median():.2f}')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(f'{col}: Mean={train[col].mean():.2f}, Median={train[col].median():.2f}, Std={train[col].std():.2f}')


## 5. Temporal Analysis

In [ ]:
date_cols = [c for c in train.columns if 'dt' in c.lower() or 'date' in c.lower()]
for col in date_cols:
    train[col] = pd.to_datetime(train[col], errors='coerce')
    if train[col].notna().sum() > 0:
        print(f'{col}: {train[col].min()} to {train[col].max()}')
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        train[col].dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
        axes[0].set_title('Day of Week')
        axes[0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
        
        train[col].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Month')
        
        train.groupby(train[col].dt.date).size().plot(ax=axes[2], color='teal')
        axes[2].set_title('Daily Volume')
        axes[2].tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Store Analysis

In [ ]:
store_cols = [c for c in train.columns if 'store' in c.lower() or 'city' in c.lower() or 'warehouse' in c.lower()]
for col in store_cols:
    print(f'\n{col} — {train[col].nunique()} unique')
    fig, ax = plt.subplots(figsize=(14, 5))
    train[col].value_counts().head(20).plot(kind='bar', ax=ax, color=sns.color_palette('viridis', 20))
    ax.set_title(f'{col} Distribution')
    plt.tight_layout()
    plt.show()


## 7. Weather & External Factors

In [ ]:
weather_cols = [c for c in train.columns if any(k in c.lower() for k in ['weather','temp','humid','wind','precip','holiday','activity'])]
print('Weather/external columns:', weather_cols)

num_w = [c for c in weather_cols if train[c].dtype in ['float64','int64']]
n = len(num_w)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(num_w):
        if train[col].nunique() <= 5:
            train[col].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='coral')
        else:
            axes[i].hist(train[col].dropna(), bins=40, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col}')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 8. Discount & Promotion Impact

In [ ]:
disc_cols = [c for c in train.columns if 'discount' in c.lower() or 'promo' in c.lower() or 'activity' in c.lower()]
sale_col = [c for c in train.columns if 'sale_amount' in c.lower()]

if disc_cols and sale_col:
    sc = sale_col[0]
    for col in disc_cols:
        if train[col].dtype in ['float64','int64']:
            fig, ax = plt.subplots(figsize=(12, 5))
            if train[col].nunique() <= 10:
                train.groupby(col)[sc].mean().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
                ax.set_title(f'Avg {sc} by {col}')
                ax.set_ylabel(f'Mean {sc}')
            else:
                ax.scatter(train[col], train[sc], alpha=0.1, s=5)
                ax.set_title(f'{sc} vs {col}')
                ax.set_xlabel(col)
                ax.set_ylabel(sc)
            plt.tight_layout()
            plt.show()


## 9. Hourly Patterns (if hourly arrays exist)

In [ ]:
hourly_cols = [c for c in train.columns if 'hourly' in c.lower() or 'hour' in c.lower()]
print('Hourly columns:', hourly_cols)

if hourly_cols:
    # Try to parse first hourly column
    col = hourly_cols[0]
    sample_val = train[col].iloc[0]
    print(f'Sample value type: {type(sample_val)}')
    print(f'Sample value: {str(sample_val)[:200]}')
    
    # If it is a string representation of a list
    try:
        parsed = train[col].head(1000).apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        hourly_df = pd.DataFrame(parsed.tolist())
        avg_hourly = hourly_df.mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.bar(range(len(avg_hourly)), avg_hourly.values, color=sns.color_palette('coolwarm', len(avg_hourly)))
        ax.set_title(f'Average {col} by Hour')
        ax.set_xlabel('Hour')
        ax.set_ylabel('Value')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Could not parse hourly data: {e}')


## 10. Product Distribution

In [ ]:
prod_col = [c for c in train.columns if 'product' in c.lower()]
if prod_col:
    pc = prod_col[0]
    print(f'{pc}: {train[pc].nunique()} unique products')
    
    top = train[pc].value_counts().head(30)
    fig, ax = plt.subplots(figsize=(14, 8))
    top.plot(kind='barh', ax=ax, color=sns.color_palette('magma', 30))
    ax.set_title(f'Top 30 Products by Record Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


## 11. Correlation Matrix

In [ ]:
num_df = train.select_dtypes(include=[np.number])
# Exclude ID columns
id_cols = [c for c in num_df.columns if 'id' in c.lower()]
num_df = num_df.drop(columns=id_cols, errors='ignore')

if num_df.shape[1] > 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()


## 12. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- Fresh/perishable retail — critical category for shelf management
- 4.5M+ records for robust ML model training
- Hourly sales/stock arrays — granular demand patterns for facing optimization
- Weather, holiday, activity flags — external factor impact on demand
- Multi-store (26 stores) — localized planogram potential
- Discount data — promotion impact analysis

**Limitations:**
- Fresh/perishable focus may not generalize to all grocery categories
- No physical shelf layout or aisle data
- No basket-level co-purchase information
- Single city — limited geographic diversity
